<a href="https://colab.research.google.com/github/lareadeola/Machine-Learning/blob/main/Burner_Sim_Research_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ST_DBSCAN

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 422.7/422.7 kB 7.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for ST_DBSCAN: filename=st_dbscan-0.2.3-py2.py3-none-any.whl size=6926 sha256=65cb169559f416acbbbaca64c67c531bca366411d926774a743b8c8309c85d53
  Stored in directory: /root/.cache/pip/wheels/9d/7d/ca/b02aac4bddf41dfd922d6e9ec1eda5b8dd2e08ab5fe1b37adb
Successfully built ST_DBSCAN


In [2]:
import pandas as pd
import numpy as np
from st_dbscan import ST_DBSCAN
from sklearn.metrics import v_measure_score, adjusted_rand_score

In [3]:
# 1. Load the dataset we generated
df = pd.read_csv("burner_sim_research_dataset.csv")

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Timestamp       10000 non-null  datetime64[ns]
 1   IMSI            10000 non-null  int64         
 2   IMEI            10000 non-null  int64         
 3   Cell_ID         10000 non-null  int64         
 4   LAC             10000 non-null  int64         
 5   Behavior_Label  10000 non-null  object        
 6   Time_Epoch      10000 non-null  int64         
dtypes: datetime64[ns](1), int64(5), object(1)
memory usage: 547.0+ KB


In [4]:
# 2. Preprocess Time into a numerical float value (Epoch timestamps in seconds)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Time_Epoch'] = df['Timestamp'].astype('int64') // 10**9

In [5]:
# 3. Extract features for ST-DBSCAN
# The algorithm expects: [Longitude, Latitude, Time_Value]
X = df[['lon', 'lat', 'Time_Epoch']].values

KeyError: "['lon', 'lat'] not in index"

In [ ]:
# 4. Initialize ST-DBSCAN
# Adjust hyper-parameters based on tactical assumptions:
# eps1 = 0.15 (approx. 15km spatial radius)
# eps2 = 14400 (4 hours temporal gap window in seconds)
st_dbscan = ST_DBSCAN(eps1=0.15, eps2=14400, min_samples=4)

print("Running Spatial-Temporal Clustering...")
st_dbscan.fit(X)

In [ ]:
# 5. Extract calculated cluster assignments
df['Cluster_Labels'] = st_dbscan.labels_

# Note: Points labeled '-1' are categorized as Noise (e.g., normal civilian entries)
print(f"Discovered Clusters: {len(set(df['Cluster_Labels'])) - (1 if -1 in df['Cluster_Labels'] else 0)}")
print(f"Points flagged as civilian background noise: {list(df['Cluster_Labels']).count(-1)}")

In [ ]:
# =====================================================================
# STEP 6: THESIS VALIDATION (Evaluating Unsupervised vs Ground-Truth)
# =====================================================================
# Map true identity tracking by grouping rows by their hidden hardware IMEI
# This creates a baseline of what the true target paths look like.
df['True_ID_Label'] = df['IMEI'].astype('category').cat.codes

# Filter out rows that were true civilians to measure how well it unmasked the targets
adversarial_only_df = df[df['Behavior_Label'] == 'Adversarial_Burner_Swap']

ari_score = adjusted_rand_score(adversarial_only_df['True_ID_Label'], adversarial_only_df['Cluster_Labels'])
v_score = v_measure_score(adversarial_only_df['True_ID_Label'], adversarial_only_df['Cluster_Labels'])

print("\n--- Model Performance Evaluation ---")
print(f"Adjusted Rand Index (ARI): {ari_score:.4f}  (Closer to 1.0 is perfect tracking)")
print(f"V-Measure Score: {v_score:.4f}            (Measures cluster completeness)")

In [12]:
import pandas as pd
import numpy as np
from st_dbscan import ST_DBSCAN
from sklearn.metrics import v_measure_score, adjusted_rand_score

# =====================================================================
# STEP 1: INFRASTRUCTURE MERGE (Injecting Lat/Lon)
# =====================================================================
print("Loading research datasets...")
# Load your generated log dataset
df_logs = pd.read_csv("burner_sim_research_dataset_2.csv")

# Load your clean OpenCellID infrastructure map
df_masts = pd.read_csv("nigeria_cell_towers.csv")

print("Merging logs with OpenCellID coordinates...")
# OpenCellID uses 'cell' and 'area' for Cell ID and LAC. Match them up.
df = pd.merge(
    df_logs,
    df_masts[['cell', 'area', 'lon', 'lat']],
    left_on=['Cell_ID', 'LAC'],
    right_on=['cell', 'area'],
    how='inner'
)

# Drop redundant lookup columns post-merge
df = df.drop(columns=['cell', 'area'])

# =====================================================================
# STEP 2: TEMPORAL FEATURE ENGINEERING
# =====================================================================
# Transform standard timestamp strings into float Epoch values (seconds)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Time_Epoch'] = df['Timestamp'].astype('int64') // 10**9

# =====================================================================
# STEP 3: FEATURE MATRIX SELECTION
# =====================================================================
# ST-DBSCAN expects an n x 3 matrix in the exact order: [Longitude, Latitude, Time]
X = df[['lon', 'lat', 'Time_Epoch']].values

Loading research datasets...
Merging logs with OpenCellID coordinates...


In [14]:
# Extract feature array post-merge
df['True_ID_Label'] = df['IMEI'].astype('category').cat.codes

print("\n--- Hyperparameter Grid Search Matrix Optimization ---")
# Spatial Epsilon options (in degrees: 0.05 is roughly 5.5km, 0.1 is 11km)
for test_eps1 in [0.05, 0.1, 0.2]:
    # Temporal Epsilon options (in seconds: 14400 = 4 hours, 28800 = 8 hours)
    for test_eps2 in [14400, 28800]:

        model = ST_DBSCAN(eps1=test_eps1, eps2=test_eps2, min_samples=4)
        model.fit(X)
        df['Cluster_Labels'] = model.labels

        # Isolate target rows to compute evaluation validation metrics
        adv_subset = df[df['Behavior_Label'] == 'Adversarial_Burner_Swap']

        v_score = v_measure_score(adv_subset['True_ID_Label'], adv_subset['Cluster_Labels'])
        ari_score = adjusted_rand_score(adv_subset['True_ID_Label'], adv_subset['Cluster_Labels'])
        n_clusters = len(set(df['Cluster_Labels'])) - (1 if -1 in df['Cluster_Labels'] else 0)

        print(f"Parameters: [eps1={test_eps1}, eps2={test_eps2}] -> Clusters Found: {n_clusters} | V-Measure: {v_score:.4f} | ARI: {ari_score:.4f}")


--- Hyperparameter Grid Search Matrix Optimization ---
Parameters: [eps1=0.05, eps2=14400] -> Clusters Found: 101 | V-Measure: 0.0077 | ARI: 0.0000
Parameters: [eps1=0.05, eps2=28800] -> Clusters Found: 101 | V-Measure: 0.0077 | ARI: 0.0000
Parameters: [eps1=0.1, eps2=14400] -> Clusters Found: 279 | V-Measure: 0.0173 | ARI: 0.0000
Parameters: [eps1=0.1, eps2=28800] -> Clusters Found: 279 | V-Measure: 0.0173 | ARI: 0.0000
Parameters: [eps1=0.2, eps2=14400] -> Clusters Found: 529 | V-Measure: 0.0388 | ARI: 0.0000
Parameters: [eps1=0.2, eps2=28800] -> Clusters Found: 529 | V-Measure: 0.0388 | ARI: 0.0000


In [15]:
# =====================================================================
# STEP 4: ST-DBSCAN MODEL EXECUTION
# =====================================================================
# eps1 = 0.15 (spatial constraint degrees ~ 15km)
# eps2 = 14400 (temporal constraint window: 4 hours in seconds)
st_dbscan = ST_DBSCAN(eps1=0.05, eps2=14400, min_samples=4)

print(f"Running ST-DBSCAN clustering across {len(X)} records...")
st_dbscan.fit(X)

# Assign cluster outcomes back to our master dataframe
df['Cluster_Labels'] = st_dbscan.labels

Running ST-DBSCAN clustering across 10002 records...


In [16]:
# =====================================================================
# STEP 5: FORENSIC PERFORMANCE EVALUATION
# =====================================================================
# Generate numeric codes from our hidden ground-truth IMEI marker
df['True_ID_Label'] = df['IMEI'].astype('category').cat.codes

# Isolate the adversarial subset to track algorithm execution accuracy
adversarial_only = df[df['Behavior_Label'] == 'Adversarial_Burner_Swap']

if len(adversarial_only) > 0:
    ari_score = adjusted_rand_score(adversarial_only['True_ID_Label'], adversarial_only['Cluster_Labels'])
    v_score = v_measure_score(adversarial_only['True_ID_Label'], adversarial_only['Cluster_Labels'])

    print("\n--- Model Metrics Execution Output ---")
    print(f"Discovered Clusters: {len(set(df['Cluster_Labels'])) - (1 if -1 in df['Cluster_Labels'] else 0)}")
    print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")
    print(f"V-Measure Tracking Score: {v_score:.4f}")
else:
    print("Warning: No adversarial samples matched coordinate bounds.")


--- Model Metrics Execution Output ---
Discovered Clusters: 101
Adjusted Rand Index (ARI): 0.0000
V-Measure Tracking Score: 0.0077


Experiment 3

In [17]:
import pandas as pd
import numpy as np
from st_dbscan import ST_DBSCAN
from sklearn.metrics import v_measure_score, adjusted_rand_score

# =====================================================================
# STEP 1: LOAD AND CLEAN INFRASTRUCTURE LOOKUP TABLE
# =====================================================================
print("Loading infrastructure data...")
df_masts = pd.read_csv("nigeria_cell_towers.csv").dropna(subset=['lon', 'lat'])

# CRITICAL FIX: Deduplicate the tower database so each Cell/LAC key is unique.
# We take the mean coordinates if duplicates exist to provide a single anchor point.
df_masts_unique = df_masts.groupby(['cell', 'area'])[['lon', 'lat']].mean().reset_index()

# =====================================================================
# STEP 2: LOAD LOGS AND EXECUTE CLEAN MERGE
# =====================================================================
print("Loading continuous path logs...")
df_logs = pd.read_csv("burner_sim_research_dataset_2.csv")

print("Merging logs with deduplicated coordinates...")
df = pd.merge(
    df_logs,
    df_masts_unique,
    left_on=['Cell_ID', 'LAC'],
    right_on=['cell', 'area'],
    how='inner'
)
df = df.drop(columns=['cell', 'area'])

# Save the clean merged dataset for verification
df.to_csv("merged_burner_sim_geo_dataset.csv", index=False)

# =====================================================================
# STEP 3: TEMPORAL FEATURE ENGINEERING
# =====================================================================
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Time_Epoch'] = df['Timestamp'].astype('int64') // 10**9

# Prepare features [Longitude, Latitude, Time_Epoch]
X = df[['lon', 'lat', 'Time_Epoch']].values
df['True_ID_Label'] = df['IMEI'].astype('category').cat.codes

# =====================================================================
# STEP 4: HYPERPARAMETER OPTIMIZATION SWEEP
# =====================================================================
print("\n--- Optimizing ST-DBSCAN Over Continuous Space ---")

Loading infrastructure data...
Loading continuous path logs...
Merging logs with deduplicated coordinates...

--- Optimizing ST-DBSCAN Over Continuous Space ---


In [26]:
# Let's test broader parameters to accommodate the movement scale
for test_eps1 in [0.1, 0.5, 1.0]: # Spatial Epsilon (Degrees)
    for test_eps2 in [28800, 86400]: # Temporal Epsilon (8 hours, 24 hours in seconds)

        st_dbscan = ST_DBSCAN(eps1=test_eps1, eps2=test_eps2, min_samples=6)
        st_dbscan.fit(X)
        df['Cluster_Labels'] = st_dbscan.labels

        # Isolate target rows to compute evaluation validation metrics
        adv_subset = df[df['Behavior_Label'] == 'Adversarial_Burner_Swap']

        v_score = v_measure_score(adv_subset['True_ID_Label'], adv_subset['Cluster_Labels'])
        ari_score = adjusted_rand_score(adv_subset['True_ID_Label'], adv_subset['Cluster_Labels'])
        n_clusters = len(set(df['Cluster_Labels'])) - (1 if -1 in df['Cluster_Labels'] else 0)

        print(f"Params: [eps1={test_eps1}, eps2={test_eps2}] -> Clusters Found: {n_clusters} | V-Measure: {v_score:.4f} | ARI: {ari_score:.4f}")

Params: [eps1=0.1, eps2=28800] -> Clusters Found: 36 | V-Measure: 0.0042 | ARI: -0.0000
Params: [eps1=0.1, eps2=86400] -> Clusters Found: 36 | V-Measure: 0.0042 | ARI: -0.0000
Params: [eps1=0.5, eps2=28800] -> Clusters Found: 377 | V-Measure: 0.0271 | ARI: 0.0000
Params: [eps1=0.5, eps2=86400] -> Clusters Found: 377 | V-Measure: 0.0271 | ARI: 0.0000
Params: [eps1=1.0, eps2=28800] -> Clusters Found: 519 | V-Measure: 0.1340 | ARI: 0.0045
Params: [eps1=1.0, eps2=86400] -> Clusters Found: 519 | V-Measure: 0.1340 | ARI: 0.0045


In [27]:
# =====================================================================
# STEP 4: ST-DBSCAN MODEL EXECUTION
# =====================================================================
# eps1 = 0.15 (spatial constraint degrees ~ 15km)
# eps2 = 14400 (temporal constraint window: 4 hours in seconds)
st_dbscan = ST_DBSCAN(eps1=0.1, eps2=86400, min_samples=6)

print(f"Running ST-DBSCAN clustering across {len(X)} records...")
st_dbscan.fit(X)

# Assign cluster outcomes back to our master dataframe
df['Cluster_Labels'] = st_dbscan.labels

Running ST-DBSCAN clustering across 10000 records...


In [28]:
# =====================================================================
# STEP 5: FORENSIC PERFORMANCE EVALUATION
# =====================================================================
# Generate numeric codes from our hidden ground-truth IMEI marker
df['True_ID_Label'] = df['IMEI'].astype('category').cat.codes

# Isolate the adversarial subset to track algorithm execution accuracy
adversarial_only = df[df['Behavior_Label'] == 'Adversarial_Burner_Swap']

if len(adversarial_only) > 0:
    ari_score = adjusted_rand_score(adversarial_only['True_ID_Label'], adversarial_only['Cluster_Labels'])
    v_score = v_measure_score(adversarial_only['True_ID_Label'], adversarial_only['Cluster_Labels'])

    print("\n--- Model Metrics Execution Output ---")
    print(f"Discovered Clusters: {len(set(df['Cluster_Labels'])) - (1 if -1 in df['Cluster_Labels'] else 0)}")
    print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")
    print(f"V-Measure Tracking Score: {v_score:.4f}")
else:
    print("Warning: No adversarial samples matched coordinate bounds.")


--- Model Metrics Execution Output ---
Discovered Clusters: 36
Adjusted Rand Index (ARI): -0.0000
V-Measure Tracking Score: 0.0042


Experiment 4

In [37]:
import pandas as pd
import numpy as np
from st_dbscan import ST_DBSCAN
from sklearn.metrics import v_measure_score, adjusted_rand_score

# =====================================================================
# STEP 1: LOAD AND CLEAN INFRASTRUCTURE LOOKUP TABLE
# =====================================================================
print("Loading infrastructure data...")
df_masts = pd.read_csv("nigeria_cell_towers.csv").dropna(subset=['lon', 'lat'])

# CRITICAL FIX: Deduplicate the tower database so each Cell/LAC key is unique.
# We take the mean coordinates if duplicates exist to provide a single anchor point.
df_masts_unique = df_masts.groupby(['cell', 'area'])[['lon', 'lat']].mean().reset_index()

# =====================================================================
# STEP 2: LOAD LOGS AND EXECUTE CLEAN MERGE
# =====================================================================
print("Loading continuous path logs...")
df_logs = pd.read_csv("burner_sim_research_dataset_2.csv")

print("Merging logs with deduplicated coordinates...")
df = pd.merge(
    df_logs,
    df_masts_unique,
    left_on=['Cell_ID', 'LAC'],
    right_on=['cell', 'area'],
    how='inner'
)
df = df.drop(columns=['cell', 'area'])

# Save the clean merged dataset for verification
df.to_csv("merged_burner_sim_geo_dataset.csv", index=False)

# =====================================================================
# STEP 3: TEMPORAL FEATURE ENGINEERING
# =====================================================================
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Time_Epoch'] = df['Timestamp'].astype('int64') // 10**9

# Prepare features [Longitude, Latitude, Time_Epoch]
X = df[['lon', 'lat', 'Time_Epoch']].values
df['True_ID_Label'] = df['IMEI'].astype('category').cat.codes

# =====================================================================
# STEP 4: HYPERPARAMETER OPTIMIZATION SWEEP
# =====================================================================
print("\n--- Optimizing ST-DBSCAN Over Continuous Space ---")

Loading infrastructure data...
Loading continuous path logs...
Merging logs with deduplicated coordinates...

--- Optimizing ST-DBSCAN Over Continuous Space ---


In [38]:
# =====================================================================
# STEP 3 (REVISED): COMPREHENSIVE VECTOR SCALING
# =====================================================================
from sklearn.preprocessing import MinMaxScaler

# Isolate raw features
X_raw = df[['lon', 'lat', 'Time_Epoch']].values

# Scale all three features strictly between 0 and 1
# This levels the playing field between degrees (0.1) and seconds (14,400)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_raw)

# =====================================================================
# STEP 4 (REVISED): EXECUTE MODEL ON SCALED MATRIX
# =====================================================================
# Because the feature space is now bounded between 0 and 1,
# your epsilon values must be small fractions.
print("Running ST-DBSCAN across balanced features...")

for test_eps1 in [0.01, 0.03, 0.05]:       # Normalized Spatial Step bounds
    for test_eps2 in [0.01, 0.05, 0.1]:    # Normalized Temporal Step bounds
      for test_samples in [4, 6, 8]:       # min samples bounds

        st_dbscan = ST_DBSCAN(eps1=test_eps1, eps2=test_eps2, min_samples=test_samples)
        st_dbscan.fit(X_scaled)

        df['Cluster_Labels'] = st_dbscan.labels
        adv_subset = df[df['Behavior_Label'] == 'Adversarial_Burner_Swap']

        v_score = v_measure_score(adv_subset['True_ID_Label'], adv_subset['Cluster_Labels'])
        ari_score = adjusted_rand_score(adv_subset['True_ID_Label'], adv_subset['Cluster_Labels'])
        n_clusters = len(set(df['Cluster_Labels'])) - (1 if -1 in df['Cluster_Labels'] else 0)

        print(f"Scaled Params: [eps1={test_eps1}, eps2={test_eps2}, min_samples={test_samples}] -> Clusters: {n_clusters} | V-Measure: {v_score:.4f} | ARI: {ari_score:.4f}")

Running ST-DBSCAN across balanced features...
Scaled Params: [eps1=0.01, eps2=0.01, min_samples=4] -> Clusters: 452 | V-Measure: 0.9432 | ARI: 0.6961
Scaled Params: [eps1=0.01, eps2=0.01, min_samples=6] -> Clusters: 271 | V-Measure: 0.9425 | ARI: 0.6955
Scaled Params: [eps1=0.01, eps2=0.01, min_samples=8] -> Clusters: 221 | V-Measure: 0.6592 | ARI: 0.1022
Scaled Params: [eps1=0.01, eps2=0.05, min_samples=4] -> Clusters: 449 | V-Measure: 0.9257 | ARI: 0.6467
Scaled Params: [eps1=0.01, eps2=0.05, min_samples=6] -> Clusters: 297 | V-Measure: 0.9249 | ARI: 0.6455
Scaled Params: [eps1=0.01, eps2=0.05, min_samples=8] -> Clusters: 288 | V-Measure: 0.7080 | ARI: 0.1645
Scaled Params: [eps1=0.01, eps2=0.1, min_samples=4] -> Clusters: 383 | V-Measure: 0.8886 | ARI: 0.5282
Scaled Params: [eps1=0.01, eps2=0.1, min_samples=6] -> Clusters: 270 | V-Measure: 0.8881 | ARI: 0.5277
Scaled Params: [eps1=0.01, eps2=0.1, min_samples=8] -> Clusters: 283 | V-Measure: 0.7153 | ARI: 0.1859
Scaled Params: [eps1=